# Analyze Traces with SQL and Litefuse

**Module 7: Trace Analysis and Litefuse** · [Quiz](quiz07_analysis_and_litefuse.ipynb) · [Course README](../../README.md)

In this Lab you analyze traces in two ways. First you query the spans that the services of the OpenTelemetry demo write to `otel_traces`: you rank the slowest requests of each service with a window function and summarize latency with percentiles. Then you trace a small application built on a large language model (LLM) with Litefuse, an LLM observability platform that keeps its data in Doris. You send the application's traces with the Langfuse Python SDK, find them in Doris with SQL, and open the same traces in Litefuse's web UI.

The application calls a simulated model in the notebook: it needs no API key or account, and nothing leaves your computer.

Plan about 45 minutes: 15 for sections 1 and 2, 30 for sections 3 to 9. You need the DOG stack and the demo from Module 1. Section 4 starts Litefuse; its first start downloads about 1.2 GB of images, which takes a few minutes more.

### Initialize the Lab

Run this cell once before using `lab.shell(...)` or `lab.sql(...)`.


In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "doris_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from doris_course import DorisLab

lab = DorisLab(lab_dir=COURSE_ROOT)


## 1. Check the spans in Doris

The queries below rank and summarize server spans. A server span covers the work that a service does for one incoming request; its `span_kind` is `SPAN_KIND_SERVER`. Count them by service:

In [ ]:
server_spans = lab.wait_for_rows(
    """
    SELECT service_name, COUNT(*) AS server_spans, MAX(timestamp) AS latest_span
    FROM otel_traces
    WHERE span_kind = 'SPAN_KIND_SERVER'
    GROUP BY service_name
    ORDER BY server_spans DESC
    """,
    title="Server spans by service in otel_traces",
    hint="otel_traces has no server spans. Start the demo with lab.start_demo() in Module 1, "
         "wait a minute, then rerun this cell.",
)

**Expected result**

One row per service that handles requests, about 14 once the demo has run for a few minutes. `frontend-proxy` and `frontend` have the most server spans; `checkout`, `payment`, and `email` have few, because the load generator browses far more often than it places orders. The span that you sent in Module 2 is not listed: it is an internal span, not a server span.

## 2. Rank and summarize spans with SQL

Doris provides a rich set of window functions and analytical functions. The following simple examples demonstrate how to use typical window functions and analytical functions.

When processing observability trace data, long-lived connections and streaming services often contain extremely dense arrays of internal events. How can Doris calculate the top three longest-running spans for each `service_name` and count the number of events contained in each span?

The `duration` column holds the length of a span in microseconds, the same as `end_time` minus `timestamp`, so the queries divide it by 1,000 to show milliseconds.

In [ ]:
lab.sql(r"""
WITH ranked_traces AS (
    SELECT
        service_name,
        trace_id,
        span_id,
        span_name,
        timestamp,
        ROUND(duration / 1000.0, 2) AS duration_ms,
        size(events) AS event_count,
        ROW_NUMBER() OVER (
            PARTITION BY service_name
            ORDER BY duration DESC
        ) AS rank_num
    FROM
        otel_traces
    WHERE
        span_kind = 'SPAN_KIND_SERVER'
)
SELECT
    service_name,
    trace_id,
    span_id,
    span_name,
    timestamp,
    duration_ms,
    event_count,
    rank_num
FROM
    ranked_traces
WHERE
    rank_num <= 3
ORDER BY
    service_name,
    rank_num;
""", title='Longest server spans by service')


**Expected result**

Up to three rows for each service, ranked by `duration_ms`. For most services, the longest requests take from about a millisecond to about a hundred, such as `PlaceOrder` in `checkout`. The `EventStream` spans of `flagd` are much longer, from seconds up to about 600,000 ms: `EventStream` is a streaming call that stays open for up to 10 minutes, so its duration measures how long the stream was open, not how slow a request was. `event_count` shows how many events a span recorded, for example 5 for `PlaceOrder`.

First, `size(events)` counts the number of elements in the array, which is the number of message events. The query then partitions rows by `service_name` and assigns a sequential rank starting from 1 based on `duration` in descending order. The outer query uses `rank_num <= 3` to obtain the three longest-running records for each service.

This is a familiar use of window functions.

What if you need to calculate P90 and P99 response latency, maximum latency, and the average number of events per request by service and API? How can this requirement be implemented in Doris?

This requirement is a straightforward aggregate query:

In [ ]:
lab.sql(r"""
SELECT
    service_name,
    span_name,
    COUNT(*) AS total_spans,

    -- 1. Basic duration analysis (convert microseconds to milliseconds)
    ROUND(AVG(duration) / 1000.0, 2) AS avg_duration_ms,
    ROUND(MAX(duration) / 1000.0, 2) AS max_duration_ms,

    -- 2. Duration percentile analysis (P90 / P99)
    ROUND(PERCENTILE_APPROX(duration, 0.90) / 1000.0, 2) AS p90_duration_ms,
    ROUND(PERCENTILE_APPROX(duration, 0.99) / 1000.0, 2) AS p99_duration_ms,

    -- 3. Average event count per request
    ROUND(AVG(size(events)), 1) AS avg_event_count
FROM
    otel_traces
WHERE
    span_kind = 'SPAN_KIND_SERVER'
GROUP BY
    service_name,
    span_name
ORDER BY
    p99_duration_ms DESC;
""", title='Latency and event-count summary')


**Expected result**

One row per service and span name, sorted by `p99_duration_ms`. The streaming calls come first: `EventStream` in `flagd`, and `ingress` in `frontend-proxy`, whose slowest spans are the same flag streams passing through the proxy. Below them, the checkout path (`POST /api/checkout` in `frontend` and `PlaceOrder` in `checkout`) usually has the highest P99, and lookups such as `GetProduct` take well under a millisecond. Check `total_spans` as well: the checkout path has only a few dozen spans, so a handful of slow requests sets its P99.

## 3. Litefuse: traces of an LLM application

The spans in `otel_traces` describe requests between services: which service called which, and how long each call took. An application built on an LLM needs more about each request: the prompt that it sent to the model, the model's answer, how many tokens the model counted and what they cost, and the steps around the model call, such as the search for documents to put in the prompt. LLM observability platforms record a trace for each request too, with an observation for each step.

Litefuse is such a platform. It is an open-source fork of Langfuse ([github.com/litefuse/litefuse](https://github.com/litefuse/litefuse), MIT license): Langfuse keeps its traces in ClickHouse, Litefuse keeps them in Apache Doris. Its API is compatible with Langfuse's, so applications send their traces with Langfuse's SDKs, pointed at Litefuse's address. In this course, Litefuse keeps its tables in the DOG stack's Doris, in the database `litefuse`, next to `otel`.

Litefuse 26.3.0 runs as five containers:

| Container | What it does |
| --- | --- |
| `doris-obs-litefuse-web` | The web UI, and the API that receives the traces |
| `doris-obs-litefuse-worker` | Loads the traces into Doris in the background |
| `doris-obs-litefuse-postgres` | PostgreSQL: users, organizations, projects, API keys, and prompts |
| `doris-obs-litefuse-redis` | Redis: the queues between the web container and the worker |
| `doris-obs-litefuse-silo` | Silo, an S3-compatible object storage server forked from MinIO: each batch of traces that the API receives |

The API keeps each batch that it receives in object storage and queues it. The worker loads the queued batches into Doris with Stream Load, and the UI and the API read the traces back from Doris with SQL.

## 4. Start Litefuse

`lab.start_litefuse()` starts Litefuse with the Compose file `environments/litefuse/compose.litefuse.yml`, which pins the version of every image. Litefuse starts with a user, an organization `Doris course`, and a project `LLM app` with a pair of API keys, so you do not sign up.

Before it starts the containers, the cell runs one statement on Doris, so that Litefuse can load data:

- Litefuse loads its data with Stream Load through the FE, at `doris-obs:8030`. The FE does not take the data itself: it answers with a redirect to a BE, at the address that the BE registered with.
- The BE of this single-container Doris registered as `127.0.0.1`. From another container, that address is the container itself, and the load fails with "connection refused".
- `ALTER SYSTEM MODIFY BACKEND` gives the BE a private endpoint, the tag `private_endpoint`. When a BE has one, the FE redirects a Stream Load to the host that the client used to reach the FE, here `doris-obs`, with the port of the endpoint, 8040. Containers on the `doris-obs` network reach the BE there. [Stream Load Practices in Complex Network Environments](https://doris.apache.org/docs/4.x/data-operate/import/load-best-practices/stream-load-in-complex-network/) describes the endpoints.

In [ ]:
lab.start_litefuse()

**Expected result**

In order:

- The port check: 23100 (`DORIS_OBS_LITEFUSE_PORT`) is `available`, or `used by this course` when Litefuse already runs.
- The statement `ALTER SYSTEM MODIFY BACKEND "127.0.0.1:9050"` with the tags `location` and `private_endpoint`, and "Statement succeeded". `SHOW BACKENDS` now shows both tags in its `Tag` column.
- The log of `docker compose`: on the first run, the downloads, then five `doris-obs-litefuse-*` containers, which `ps` lists as running; Postgres, Redis, and Silo also as healthy.
- The project that the API keys belong to: the organization `Doris course`, the project `LLM app`, and its ID, `llm_app`.
- The card "Litefuse is running", about 10 seconds after the containers start: on its first start, Litefuse creates its database in Postgres and its tables in Doris before it answers.

Rerunning the cell keeps the running containers and their data.

## 5. Trace a small LLM application

The next cell builds a small assistant that answers questions about this course, and traces it with the Langfuse Python SDK (`langfuse`, installed with the course's requirements). Each step of the assistant is a function with the `@observe` decorator, which records each call as an observation, with the function's arguments as its input and its return value as its output:

- `answer_question` handles one request and is the root of its trace. `propagate_attributes` names the trace `course-assistant` and records the user and the session that asked; the observations below it inherit them.
- `search_docs` looks up the documents about the topic of the question: an observation of type `retriever`.
- `call_model` stands for the call to the model: an observation of type `generation`. The simulated model returns a prepared answer, counts one token per word, and writes 100 tokens per second. `update_current_generation` records the prompt (the chat messages), the model, the token counts, and the cost of the call, from an example price of 0.01 USD per 1,000 input tokens and 0.03 USD per 1,000 output tokens. With a real model, Langfuse's integrations, such as the one for the OpenAI SDK, record these by themselves: [Optional Lab 7A](optional07_trace_a_real_model.ipynb) gives this assistant a real model, on your computer or OpenAI's.

The SDK reads the API keys and Litefuse's address from the variables `LANGFUSE_PUBLIC_KEY`, `LANGFUSE_SECRET_KEY`, and `LANGFUSE_BASE_URL`. It sends the observations as OpenTelemetry spans, in batches, to Litefuse's endpoint `/api/public/otel/v1/traces`; `flush()` sends the rest before the cell ends. `OTEL_SERVICE_NAME` names the application in the spans' resource, as the demo's services are named in `otel_traces`.

In [ ]:
import os
import time

os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-doris-course"
os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-doris-course"
os.environ["LANGFUSE_BASE_URL"] = lab.litefuse_url
os.environ["OTEL_SERVICE_NAME"] = "course-assistant"

from langfuse import get_client, observe, propagate_attributes

langfuse = get_client()

# The documents that the assistant searches, by topic, and the answers of the simulated model.
DOCUMENTS = {
    "doris": "Apache Doris is a real-time analytical database. It answers SQL queries over large tables in seconds.",
    "litefuse": "Litefuse is an LLM engineering platform. It keeps its traces and observations in Apache Doris tables.",
    "collector": "The OpenTelemetry Collector receives logs, metrics, and traces and exports them to Doris.",
}
ANSWERS = {
    "What is Apache Doris?":
        "Apache Doris is a real-time analytical database that answers SQL queries over large tables in seconds.",
    "Where does Litefuse keep its traces?":
        "Litefuse keeps its traces and observations in Apache Doris tables, so you can query them with SQL.",
}
PRICE_PER_1K_TOKENS = {"input": 0.01, "output": 0.03}  # an example price list, in USD


def count_tokens(text):
    return len(text.split())  # the simulated model counts one token per word


@observe(as_type="retriever", name="search-docs")
def search_docs(question):
    time.sleep(0.05)  # a lookup in a document index takes a moment
    return [text for topic, text in DOCUMENTS.items() if topic in question.lower()]


@observe(as_type="generation", name="llm-answer")
def call_model(question, documents):
    messages = [
        {"role": "system", "content": "Answer the question with these documents: " + " ".join(documents)},
        {"role": "user", "content": question},
    ]
    answer = ANSWERS[question]
    usage = {"input": sum(count_tokens(message["content"]) for message in messages),
             "output": count_tokens(answer)}
    time.sleep(usage["output"] / 100)  # the simulated model writes 100 tokens per second
    langfuse.update_current_generation(
        input=messages,
        model="simulated-llm",
        model_parameters={"temperature": 0},
        usage_details=usage,
        cost_details={kind: round(tokens / 1000 * PRICE_PER_1K_TOKENS[kind], 6) for kind, tokens in usage.items()},
    )
    return answer


@observe(name="answer-question")
def answer_question(question, *, user_id, session_id):
    with propagate_attributes(trace_name="course-assistant", user_id=user_id, session_id=session_id):
        trace_ids.append(langfuse.get_current_trace_id())
        return call_model(question, search_docs(question))


trace_ids = []
for user_id, session_id, question in [
    ("user-1", "session-1", "What is Apache Doris?"),
    ("user-1", "session-1", "Where does Litefuse keep its traces?"),
    ("user-2", "session-2", "What is Apache Doris?"),
]:
    print(f"{user_id}: {question}\n  -> {answer_question(question, user_id=user_id, session_id=session_id)}")
langfuse.flush()
print("Trace IDs:", trace_ids)

**Expected result**

Three questions with their answers, two from `user-1` in `session-1` and one from `user-2` in `session-2`, then `Trace IDs:` and three IDs of 32 hexadecimal digits: OpenTelemetry trace IDs, like those in `otel_traces`. The next sections look up these three traces. Running the cell again sends three new traces, and the next sections then show the new ones.

## 6. Find the traces in Doris

When the first trace of a project arrives, Litefuse creates the project's two tables and names them after the project's ID, `llm_app`: `spans_llm_app` has a row per observation, and `traces_scalar_llm_app` a row per trace. The next cell waits for them.

In [ ]:
lab.wait_for_rows(
    """
    SELECT TABLE_NAME AS table_name, TABLE_TYPE AS table_type
    FROM information_schema.tables
    WHERE TABLE_SCHEMA = 'litefuse' AND TABLE_NAME LIKE '%llm_app'
    ORDER BY TABLE_NAME
    """,
    title="The tables of the project llm_app",
    hint="Litefuse creates them when the project's first trace arrives. Check that the previous cell printed "
         "three trace IDs and that Litefuse runs (section 4). The worker's log, docker logs "
         "doris-obs-litefuse-worker, shows the creation of the tables. Then rerun this cell.",
    min_rows=2,
);

**Expected result**

Two rows, `spans_llm_app` and `traces_scalar_llm_app`, both `BASE TABLE`, within a few seconds of the first traces.

In [ ]:
lab.sql("SHOW TABLES FROM litefuse", title="Litefuse's tables in Doris");

**Expected result**

Eight tables. Besides the project's two, Litefuse created six when it first started: `scores` holds scores of traces, such as user feedback or the results of evaluations; `dataset_run_items_rmt` the results of experiments on datasets; `event_log` and `blob_storage_file_log` where in object storage each event that the API received is kept; `project_environments` the environments of each project; and `schema_migrations` the migrations that Litefuse has applied to the database. Each project gets its own pair of tables for its spans and traces.

The next cell waits for the observations of the three traces. `usage_details` is a `MAP` of token counts by kind, and `TIMESTAMPDIFF` gives each observation's duration from its start and end times.

In [ ]:
TRACE_ID_LIST = ", ".join(f"'{trace_id}'" for trace_id in trace_ids)
llm_spans = lab.wait_for_rows(
    f"""
    SELECT trace_id, user_id, name, type, provided_model_name AS model, usage_details, total_cost,
           TIMESTAMPDIFF(MICROSECOND, start_time, end_time) / 1000 AS duration_ms
    FROM litefuse.spans_llm_app
    WHERE trace_id IN ({TRACE_ID_LIST})
    ORDER BY start_time, is_root DESC
    """,
    title="The observations of the three traces in spans_llm_app",
    hint="Litefuse loads a batch of traces into Doris a few seconds after it receives it. Check the worker's "
         "log with docker logs doris-obs-litefuse-worker, then rerun this cell.",
    min_rows=9,
    timeout_seconds=90,
)

**Expected result**

Nine rows, three per trace, in the order of the requests. The first time, they arrive about 10 seconds after the traces were sent: Litefuse's worker waits that long before it loads data into a project's new tables. Later runs take a few seconds. For a question about Doris:

| `name` | `type` | `model` | `usage_details` | `total_cost` | `duration_ms` |
| --- | --- | --- | --- | --- | --- |
| `answer-question` | `SPAN` | NULL | `{}` | NULL | about 220 |
| `search-docs` | `RETRIEVER` | NULL | `{}` | NULL | about 55 |
| `llm-answer` | `GENERATION` | `simulated-llm` | `{"input":26, "output":16, "total":42}` | `0.000740000000` | about 160 |

The question about Litefuse has `{"input":28, "output":17, "total":45}` and `0.000790000000`. Litefuse added the totals: `total` in `usage_details`, and `total_cost`. The root span of each trace takes about as long as its two steps together, and every row has the `user_id` that `propagate_attributes` set.

In [ ]:
lab.sql(f"""
SELECT input, output
FROM litefuse.spans_llm_app
WHERE trace_id = '{trace_ids[1]}' AND type = 'GENERATION'
""", title="The prompt and the answer of the second request");

**Expected result**

One row. `input` is the prompt: the two chat messages as JSON, the system message with the document about Litefuse that `search_docs` found, and the user's question. `output` is the answer. The other observations keep the arguments and the return value of their function in the same columns.

## 7. The same traces in Litefuse

Open Litefuse's web UI:

1. Go to [http://127.0.0.1:23100](http://127.0.0.1:23100) (or the port in `DORIS_OBS_LITEFUSE_PORT`) and sign in with the email `learner@example.com` and the password `doris-course-1`.
2. The page lists the organization **Doris course** with the project **LLM app**. Choose **Go to project**.
3. In the menu on the left, choose **Tracing**. The **Traces** tab lists the traces of the past day, newest first: the three traces named `course-assistant`, with their input, output, latency, and tokens.
4. Choose the trace whose input asks "Where does Litefuse keep its traces?". A panel opens with the trace as a tree: `course-assistant`, below it `answer-question`, and below that `search-docs` and `llm-answer`, each with its duration. Choose `llm-answer`: the panel shows the prompt as **System** and **User** messages, the answer as the **Assistant** message, the model `simulated-llm` with `temperature: 0`, the tokens `28 prompt → 17 completion (∑ 45)`, and the cost `$0.00079`.
5. In the menu, choose **Sessions**: `session-1` has two traces and `session-2` one; scroll the table to the right for their cost and tokens. **Users** lists the two users, and **Home** charts the traces, their cost by model and by user, and percentiles of their latency.

Litefuse reads all of this from the tables that you queried in section 6. Its public API does too: the next cell reads the second trace from it, with the same API keys as the SDK.

In [ ]:
trace = lab.litefuse_trace(trace_ids[1])

**Expected result**

The `curl` command, then a table with the trace's three observations: `SPAN` `answer-question`, `RETRIEVER` `search-docs`, and `GENERATION` `llm-answer` with the model `simulated-llm`, the usage `{'input': 28, 'output': 17, 'total': 45}`, and the cost `0.00079`. The title gives the trace's name, `course-assistant`, its latency, about 0.23 seconds, and its total cost, `0.00079` USD: the numbers that section 6 found in Doris.

## 8. How Litefuse uses Doris

Litefuse's tables use the features of Modules 4 to 6. The next cell lists the key columns of the project's tables.

In [ ]:
lab.sql("""
SELECT TABLE_NAME AS table_name, COLUMN_NAME AS key_column, COLUMN_KEY AS key_type
FROM information_schema.columns
WHERE TABLE_SCHEMA = 'litefuse'
  AND TABLE_NAME IN ('spans_llm_app', 'traces_scalar_llm_app')
  AND COLUMN_KEY <> ''
ORDER BY TABLE_NAME, ORDINAL_POSITION
""", title="Key columns of the project's tables");

**Expected result**

Five rows. `spans_llm_app` is a Duplicate Key table (`DUP`) on `trace_id`, `start_time`, and `span_id`: observations are only added, and the key keeps the rows of a trace together, in time order. `traces_scalar_llm_app` is a Unique Key table (`UNI`) on `id` and `start_time`: when Litefuse writes the row of a trace again, the new row replaces the old one, as the rows of `error_alerts` did in Module 4.

Both tables are partitioned by day with `AUTO PARTITION BY RANGE (date_trunc(start_time, 'day'))`, and distributed by the trace ID into `BUCKETS AUTO` (Module 5); `SHOW CREATE TABLE litefuse.spans_llm_app` shows these clauses. Litefuse also builds a synchronous materialized view on `spans_llm_app`, `trace_metrics_agg_llm_app`, which adds up the tokens and costs of each trace by day (Module 6); `SHOW CREATE MATERIALIZED VIEW trace_metrics_agg_llm_app ON litefuse.spans_llm_app` shows it.

`spans_llm_app` also has inverted indexes on 14 columns (`SHOW INDEX FROM litefuse.spans_llm_app` lists them). Those on `input` and `output` split the text into words (`"parser" = "unicode"`, in lowercase), so that a query can find the prompts and answers that contain a word without reading every row. The next cell searches the inputs for the word Litefuse.

In [ ]:
lab.sql("""
SELECT trace_id, name, type
FROM litefuse.spans_llm_app
WHERE input MATCH_ANY 'litefuse'
ORDER BY start_time, is_root DESC
""", title="Observations whose input contains the word litefuse");

**Expected result**

Three rows, the observations of the trace about Litefuse, for each time that you ran section 5: the question is in the input of all three, `answer-question` and `search-docs` received it as an argument, and `llm-answer` in its prompt. `MATCH_ANY` matches whole words in any case, and finds a row when any of the given words is in it; `MATCH_ALL` needs all of them.

## 9. Independent exercise

For each user, count the traces and add up the tokens and the cost of the model calls in `spans_llm_app`. The tokens of a call are in `usage_details['total']`. Expect two rows, `user-1` first.

Write and run your code in the next cell, then expand the reference solution.

<details>
<summary>Reference solution (expand after completing the exercise)</summary>

```python
lab.sql("""
SELECT user_id,
       COUNT(DISTINCT trace_id) AS traces,
       SUM(usage_details['total']) AS total_tokens,
       SUM(total_cost) AS total_cost_usd
FROM litefuse.spans_llm_app
WHERE type = 'GENERATION'
GROUP BY user_id
ORDER BY total_cost_usd DESC
""", title="Traces, tokens, and cost by user");
```

`user-1` has 2 traces, 87 tokens, and 0.00153 USD; `user-2` has 1 trace, 42 tokens, and 0.00074 USD. Each run of section 5 adds as much again. The **Sessions** page of Litefuse shows the same totals. Only the generations count: the other observations have no tokens or cost, and each trace has exactly one generation, so counting the traces among the generations counts them all.

</details>

## Completion criteria

- You can say what an LLM trace records about a request that a span in `otel_traces` does not.
- `spans_llm_app` has the observations of your three traces, with the model, the tokens, and the cost of each model call, and you found the same trace in Litefuse's web UI.
- You can explain why the BE needed a private endpoint before Litefuse could load data.
- Your query adds up the tokens and the cost by user.

## Clean up

Litefuse keeps running and uses about 1 GB of memory. `lab.stop_litefuse()` stops and removes its containers; its user, keys, and traces stay in its volumes and in Doris, and `lab.start_litefuse()` starts it again with them. `lab.teardown()` removes Litefuse with the rest of the environment, together with its volumes and the `litefuse` database.

## Next

Open [Quiz 7: Trace Analysis and Litefuse](quiz07_analysis_and_litefuse.ipynb), then continue to [Module 8: Alerting with Grafana](../module08-alerting/lab08_alerting.ipynb). [Optional Lab 7A: Trace a Real Model with Litefuse](optional07_trace_a_real_model.ipynb) repeats section 5 with a real model.